<a href="https://colab.research.google.com/github/Astronom2617/rossmann-sales-forecasting/blob/main/rossmann_xgboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from xgboost import XGBRegressor

train = pd.read_csv('/content/drive/MyDrive/rossmann/train.csv',
                    parse_dates=['Date'], low_memory=False)
store = pd.read_csv('/content/drive/MyDrive/rossmann/store.csv')
test = pd.read_csv('/content/drive/MyDrive/rossmann/test.csv',
                   parse_dates=['Date'])

In [16]:
train_store = train.merge(store, on='Store')
train_store_open = train_store[train_store['Open'] == 1].sort_values('Date')

In [17]:
train_store_open = train_store_open.sort_values(['Store', 'Date'])
train_store_open['lag_7'] = train_store_open.groupby('Store')['Sales'].shift(7)
train_store_open['lag_30'] = train_store_open.groupby('Store')['Sales'].shift(30)

In [18]:
train_store_open['month'] = train_store_open['Date'].dt.month
train_store_open['year'] = train_store_open['Date'].dt.year
train_store_open['day_of_week'] = train_store_open['Date'].dt.dayofweek
train_store_open.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,lag_7,lag_30,month,year,day_of_week
1014980,1,3,2013-01-02,5530,668,1,0,0,1,c,...,2008.0,0,NaN,NaN,NaN,NaN,NaN,1,2013,2
1013865,1,4,2013-01-03,4327,578,1,0,0,1,c,...,2008.0,0,NaN,NaN,NaN,NaN,NaN,1,2013,3
1012750,1,5,2013-01-04,4486,619,1,0,0,1,c,...,2008.0,0,NaN,NaN,NaN,NaN,NaN,1,2013,4
1011635,1,6,2013-01-05,4997,635,1,0,0,1,c,...,2008.0,0,NaN,NaN,NaN,NaN,NaN,1,2013,5
1009405,1,1,2013-01-07,7176,785,1,1,0,1,c,...,2008.0,0,NaN,NaN,NaN,NaN,NaN,1,2013,0


In [19]:
train_store_open = train_store_open.dropna(subset=['lag_7', 'lag_30'])
train_store_open.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,lag_7,lag_30,month,year,day_of_week
975955,1,3,2013-02-06,6140,693,1,1,0,0,c,...,2008.0,0,NaN,NaN,NaN,3725.0,5530.0,2,2013,2
974840,1,4,2013-02-07,5499,675,1,1,0,0,c,...,2008.0,0,NaN,NaN,NaN,4601.0,4327.0,2,2013,3
973725,1,5,2013-02-08,5681,630,1,1,0,0,c,...,2008.0,0,NaN,NaN,NaN,4709.0,4486.0,2,2013,4
972610,1,6,2013-02-09,5370,656,1,0,0,0,c,...,2008.0,0,NaN,NaN,NaN,5633.0,4997.0,2,2013,5
970380,1,1,2013-02-11,4409,599,1,0,0,0,c,...,2008.0,0,NaN,NaN,NaN,5970.0,7176.0,2,2013,0


In [21]:
features = ['Store', 'DayOfWeek', 'Promo', 'SchoolHoliday',
            'CompetitionDistance', 'month', 'year',
            'day_of_week', 'lag_7', 'lag_30']

train_df = train_store_open[train_store_open['Date'] < '2015-01-01']
test_df = train_store_open[train_store_open['Date'] >= '2015-01-01']

X_train = train_df[features]
y_train = train_df['Sales']

X_test = test_df[features]
y_test = test_df['Sales']

print(X_train.shape)
print(X_test.shape)

(614910, 10)
(196032, 10)


In [23]:
model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=67)
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [24]:
predictions = model.predict(X_test)

In [25]:
XGBOOST_RMSE = np.sqrt(np.mean((y_test.values - predictions)**2))
print(f"RMSE: {XGBOOST_RMSE}")

RMSE: 1632.4795967039618


### Conclusion about XGBoost
XGBoost performed best among all models. As we can see, with the addition of features such as `Store`, `DayOfWeek`, `Promo`, `SchoolHoliday`, and so on, our model achieved **the lowest RMSE (€1632)**.

| Model | RMSE |
|---|---|
| Mean per store | €3267 |
| Mean per store+day | €2014 |
| SARIMA (1,0,1)(1,0,1,7) | €2140 |
| XGBoost (100 trees) | €1632 |